# Tumor-aware Pix2Pix para síntesis T1n → T1ce (BraTS 2023-GLI)

Pipeline completo y reproducible del estudio de ablación presentado en **WAIMLAp 2026**: entrenamiento, evaluación y generación de tablas/figuras del paper.

## Protocolo experimental

- **Tarea:** traducir cortes axiales T1 nativo (T1n) → T1 con contraste (T1ce), con la máscara tumoral disponible como información auxiliar.
- **Datos:** BraTS 2023 (GLI) desde el mirror de Kaggle `luumsk/asnr-miccai-brats-2023-gli-challenge-training-data`. Se usan los **primeros 220 pacientes** (orden alfabético) y los **20 cortes centrales** de cada uno, redimensionados a 256×256 y normalizados a [-1, 1] (recorte en el percentil 99).
- **Split:** 70 % / 15 % / 15 % **por paciente** (train / val / test), con semilla fija (42) para todas las corridas. Así, la varianza entre semillas refleja solo la inicialización y el shuffle, no el split.
- **Selección de checkpoint:** mejor SSIM tumoral en **validación**. El **test se evalúa una sola vez por corrida**, al final.
- **Reproducibilidad:** `seed_everything()` siembra Python, NumPy y PyTorch; `worker_init_fn` propaga la semilla a los workers del DataLoader. **3 semillas** (42, 123, 2024) por variante.
- **Estabilización:** `EMA(decay=0.97)` en todas las variantes que lo usan, y `spectral_norm` en el discriminador desde `warmup_ema`.
- **Sin sampler ponderado:** ninguna variante usa `WeightedRandomSampler`.

## Variantes

| Variante | Descripción |
|---|---|
| `baseline` | Pix2Pix con PatchGAN estándar (sin spectral norm). Pérdida: GAN + L1 ponderado por tumor (×5). |
| `warmup_ema` | `baseline` + warm-up del generador solo con L1 (5 épocas) + EMA (0.97) + spectral norm en el discriminador. |
| `tumor_aware_disc_v2` | `warmup_ema` + discriminador condicionado a la máscara tumoral (peso de parches tumorales = 1.5). |
| `tumor_aware_ssim_freq_v2` | `tumor_aware_disc_v2` + pérdida SSIM (λ=5) + pérdida de frecuencia estilo BraSyn (λ=10). |
| `tumor_aware_disc`, `tumor_aware_ssim_freq` | Igual que las dos anteriores pero con peso de parches tumorales = 3.0. Solo para el análisis de sensibilidad (sección 10). |

## Índice

1. Preparación del entorno
2. Configuración del experimento
3. Datos
4. Modelos
5. Funciones de pérdida
6. Métricas de evaluación
7. Entrenamiento
8. Ejecución del ablation
9. Resultados: tablas y figuras
10. Análisis complementarios
- Apéndice A. Consolidación de resultados previos (opcional)

## Cómo ejecutar

1. Runtime → **GPU** (probado en T4).
2. Ejecuta las secciones **en orden**. La corrida completa es larga; es **reanudable**: si Colab se desconecta, vuelve a ejecutar desde la sección 1 y las corridas ya terminadas se saltan automáticamente.
3. *(Recomendado)* Antes de la corrida real, pon `MODO_PRUEBA = True` en la sección 2.1 para verificar en pocos minutos que todo funciona. El modo de prueba escribe en una carpeta aparte y no toca los resultados reales.

> **Artefactos generados** (en `proyecto_GAN_BraTS_v2/checkpoints_ablation_v2/` de tu Drive): checkpoints `*_best.pth`, historiales `*_historial.csv`, resultados por semilla (`resultados_*_por_seed.csv`), resumen final (`resumen_test_final.csv`) y la carpeta `figuras/` (PNG + tabla LaTeX).

## 1. Preparación del entorno
Instalación de dependencias, imports, montaje de Drive, semillas y registro de hardware/software.

In [ ]:
# ──────────────────────────────────────────────────────────────────
# [1.1] Instalar dependencias
# ──────────────────────────────────────────────────────────────────
# Qué hace : Instala las librerías que no vienen preinstaladas en Colab.
#            torch, torchvision, numpy, pandas, matplotlib y scikit-learn ya
#            están disponibles.
# ──────────────────────────────────────────────────────────────────
!pip install -q kaggle nibabel scikit-image

In [ ]:
# ──────────────────────────────────────────────────────────────────
# [1.2] Importar librerías
# ──────────────────────────────────────────────────────────────────
# Qué hace : Importa todas las librerías usadas en el notebook (un único
#            bloque de imports).
# ──────────────────────────────────────────────────────────────────
import os, glob, time, copy, random, re, gc, shutil, zipfile

import numpy as np
import pandas as pd
import nibabel as nib
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as tv_models
from torch.utils.data import Dataset, DataLoader, Subset

from sklearn.model_selection import train_test_split
from skimage.metrics import structural_similarity as ssim
from skimage.metrics import peak_signal_noise_ratio as psnr

In [ ]:
# ──────────────────────────────────────────────────────────────────
# [1.3] Montar Google Drive
# ──────────────────────────────────────────────────────────────────
# Qué hace : Monta Drive y crea la carpeta raíz del proyecto, donde se
#            guardan checkpoints, resultados y el respaldo del dataset.
# Produce  : drive_dir
# ──────────────────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

drive_dir = "/content/drive/MyDrive/proyecto_GAN_BraTS_v2"
os.makedirs(drive_dir, exist_ok=True)
print("Carpeta del proyecto en Drive:", drive_dir)

In [ ]:
# ──────────────────────────────────────────────────────────────────
# [1.4] Reproducibilidad y dispositivo
# ──────────────────────────────────────────────────────────────────
# Qué hace : Define seed_everything() (siembra Python, NumPy y PyTorch, y
#            fuerza cuDNN determinista), worker_init_fn() para los workers
#            del DataLoader, y selecciona el dispositivo.
# Produce  : seed_everything, worker_init_fn, device
# ──────────────────────────────────────────────────────────────────
def seed_everything(seed: int):
    """Siembra todas las fuentes de aleatoriedad relevantes: inicialización de
    pesos, shuffle del DataLoader y flips de augmentation."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def worker_init_fn(worker_id):
    """El módulo `random` de Python no hereda la semilla del proceso principal
    en los workers de PyTorch; sin esto, el patrón de flips no sería
    reproducible entre corridas con la misma semilla."""
    seed_worker = torch.initial_seed() % (2**32)
    np.random.seed(seed_worker)
    random.seed(seed_worker)


device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

In [ ]:
# ──────────────────────────────────────────────────────────────────
# [1.5] Información de hardware y software
# ──────────────────────────────────────────────────────────────────
# Qué hace : Imprime GPU y versiones de las librerías principales, para
#            documentar el entorno exacto de los experimentos.
# ──────────────────────────────────────────────────────────────────
import sys, sklearn, skimage, torchvision

!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

print("Python       :", sys.version.split()[0])
print("PyTorch      :", torch.__version__)
print("torchvision  :", torchvision.__version__)
print("CUDA (torch) :", torch.version.cuda)
print("cuDNN        :", torch.backends.cudnn.version())
print("numpy        :", np.__version__)
print("pandas       :", pd.__version__)
print("scikit-learn :", sklearn.__version__)
print("scikit-image :", skimage.__version__)
print("nibabel      :", nib.__version__)
import matplotlib
print("matplotlib   :", matplotlib.__version__)

## 2. Configuración del experimento
Todos los hiperparámetros, rutas y variantes se definen **aquí**, en un solo lugar. El resto del notebook solo lee estas constantes.

In [ ]:
# ──────────────────────────────────────────────────────────────────
# [2.1] Hiperparámetros, rutas y protocolo
# ──────────────────────────────────────────────────────────────────
# Qué hace : Define el modo de ejecución, las rutas, los parámetros del
#            dataset y del split, los hiperparámetros de entrenamiento y los
#            archivos de resultados.
# Produce  : constantes globales (MODO_PRUEBA, rutas, hiperparámetros, SEEDS,
#            METRICAS)
# ──────────────────────────────────────────────────────────────────
# ── Modo de ejecución ───────────────────────────────────────────────
# True  = prueba rápida (dataset reducido, 2 épocas, 1 semilla). Escribe en
#         una carpeta aparte; no toca los resultados reales.
# False = corrida real completa.
MODO_PRUEBA = False

# ── Dataset ─────────────────────────────────────────────────────────
KAGGLE_DATASET = "luumsk/asnr-miccai-brats-2023-gli-challenge-training-data"
N_PACIENTES    = 220                                   # primeros N pacientes (orden alfabético)
OUT_DIR        = "/content/brats_subset"               # extracción local (rápida)
BASE_DIR       = f"{OUT_DIR}/ASNR-MICCAI-BraTS2023-GLI-Challenge-TrainingData"
RESPALDO_ZIP   = f"{drive_dir}/brats_subset_{N_PACIENTES}_pacientes.zip"  # respaldo en Drive

MAX_CORTES = 20      # cortes axiales centrales por paciente
IMG_SIZE   = 256     # lado de la imagen (px)

# ── Split por paciente (fijo en TODAS las corridas) ─────────────────
SEED_SPLIT = 42
FRAC_TEMP  = 0.30    # val + test = 30 %, que luego se divide 50/50 → 70/15/15

# ── DataLoader ──────────────────────────────────────────────────────
BATCH_SIZE  = 8
NUM_WORKERS = 2

# ── Entrenamiento ───────────────────────────────────────────────────
LAMBDA_L1       = 25
LAMBDA_SSIM     = 5
LAMBDA_FREQ     = 10
LAMBDA_VGG      = 2.0
PESO_TUMOR      = 5.0    # peso del L1 dentro del tumor
PESO_DISC_TUMOR = 3.0    # valor por defecto del peso de parches tumorales en D
                         # (las variantes *_v2 lo sobrescriben con 1.5, ver 2.2)
EMA_DECAY       = 0.97
LR_GEN          = 2e-4
LR_DISC         = 1e-4
N_EPOCAS_WARMUP = 5
N_EPOCAS        = 40
SEEDS           = [42, 123, 2024]

if MODO_PRUEBA:
    N_EPOCAS_WARMUP, N_EPOCAS, SEEDS = 1, 2, [42]
    print("⚠️  MODO_PRUEBA activo: dataset reducido, 2 épocas, 1 semilla. "
          "Desactívalo (False) para la corrida real.")

# ── Carpetas y archivos de resultados ───────────────────────────────
CHECKPOINT_DIR = f"{drive_dir}/" + ("checkpoints_smoke_test" if MODO_PRUEBA
                                    else "checkpoints_ablation_v2")
FIG_DIR = f"{CHECKPOINT_DIR}/figuras"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)

PATH_RES_VAL    = f"{CHECKPOINT_DIR}/resultados_val_por_seed.csv"
PATH_RES_TEST   = f"{CHECKPOINT_DIR}/resultados_test_por_seed.csv"
PATH_TEST_FINAL = f"{CHECKPOINT_DIR}/resultados_test_final_4variantes.csv"
PATH_RESUMEN    = f"{CHECKPOINT_DIR}/resumen_test_final.csv"

METRICAS = ["ssim_tumor", "psnr_tumor", "ssim_global", "psnr_global"]
print("Checkpoints en:", CHECKPOINT_DIR)

In [ ]:
# ──────────────────────────────────────────────────────────────────
# [2.2] Variantes del ablation
# ──────────────────────────────────────────────────────────────────
# Qué hace : Define CONFIGS: cada variante con todos sus flags explícitos.
#            Cada una difiere de la anterior en los componentes indicados en
#            la portada. Ninguna usa sampler ponderado.
# Produce  : CONFIGS, FINAL_VARIANTS, VARIANTES_A_ENTRENAR
# ──────────────────────────────────────────────────────────────────
def _cfg(warmup, ema, sn, disc_tumor, ssim_l, freq_l, vgg_l=False, peso_disc_tumor=None):
    cfg = {
        "usar_warmup": warmup, "usar_ema": ema, "usar_spectral_norm": sn,
        "usar_disc_tumor_aware": disc_tumor, "usar_ssim_loss": ssim_l,
        "usar_freq_loss": freq_l, "usar_vgg_loss": vgg_l,
    }
    if peso_disc_tumor is not None:
        cfg["peso_disc_tumor"] = peso_disc_tumor
    return cfg


CONFIGS = {
    # ── Las 4 variantes finales del paper ───────────────────────────
    "baseline":                 _cfg(False, False, False, False, False, False),
    "warmup_ema":               _cfg(True,  True,  True,  False, False, False),
    "tumor_aware_disc_v2":      _cfg(True,  True,  True,  True,  False, False, peso_disc_tumor=1.5),
    "tumor_aware_ssim_freq_v2": _cfg(True,  True,  True,  True,  True,  True,  peso_disc_tumor=1.5),

    # ── Sensibilidad: mismas variantes con peso de parches tumorales = 3.0 ──
    "tumor_aware_disc":         _cfg(True,  True,  True,  True,  False, False, peso_disc_tumor=3.0),
    "tumor_aware_ssim_freq":    _cfg(True,  True,  True,  True,  True,  True,  peso_disc_tumor=3.0),
}

# La variante con VGG perceptual (usar_vgg_loss=True) se reportó con una sola
# corrida como hallazgo negativo (degrada por mismatch de dominio natural → MRI);
# la clase VGGPerceptualLoss se incluye (sección 5.4) pero no entra en este bucle.

FINAL_VARIANTS = ["baseline", "warmup_ema", "tumor_aware_disc_v2", "tumor_aware_ssim_freq_v2"]

# Variantes que entrena el bucle principal (sección 8). Quita las dos últimas
# para omitir el análisis de sensibilidad y ahorrar 6 corridas.
VARIANTES_A_ENTRENAR = list(CONFIGS)

print("Variantes finales     :", FINAL_VARIANTS)
print("Se entrenarán         :", VARIANTES_A_ENTRENAR, "x", len(SEEDS), "semillas")

## 3. Datos
Obtención del subset de BraTS, utilidades de lectura, clase `Dataset`, split por paciente y verificaciones de integridad.

> Los datos de BraTS **no se redistribuyen** en este repositorio; se descargan de Kaggle (requiere una cuenta y su `kaggle.json`) y se respaldan en tu Drive para no volver a descargarlos.

In [ ]:
# ──────────────────────────────────────────────────────────────────
# [3.1] Credenciales de Kaggle
# ──────────────────────────────────────────────────────────────────
# Qué hace : Configura las credenciales de Kaggle solo si hacen falta: se
#            omite si ya existe el respaldo del dataset en Drive o si
#            kaggle.json ya está configurado.
# ──────────────────────────────────────────────────────────────────
kaggle_json = os.path.expanduser("~/.kaggle/kaggle.json")

if os.path.exists(RESPALDO_ZIP):
    print("Respaldo del dataset encontrado en Drive -> no se necesita Kaggle.")
elif os.path.exists(kaggle_json):
    print("Credenciales de Kaggle ya configuradas.")
else:
    from google.colab import files
    print("Sube tu archivo kaggle.json (Kaggle -> Settings -> API -> Create New Token):")
    files.upload()                                   # se guarda como ./kaggle.json
    os.makedirs(os.path.dirname(kaggle_json), exist_ok=True)
    shutil.copy("kaggle.json", kaggle_json)
    os.chmod(kaggle_json, 0o600)
    print("Credenciales configuradas.")

In [ ]:
# ──────────────────────────────────────────────────────────────────
# [3.2] Descargar / extraer el subset de BraTS
# ──────────────────────────────────────────────────────────────────
# Qué hace : Obtiene los N_PACIENTES pacientes en OUT_DIR. Prioridad: (1) ya
#            extraído localmente, (2) respaldo en Drive, (3) descarga desde
#            Kaggle y creación del respaldo.
# Produce  : subset de pacientes en BASE_DIR
# ──────────────────────────────────────────────────────────────────
def _hay_subset_local():
    return len(glob.glob(os.path.join(BASE_DIR, "BraTS-GLI-*"))) >= N_PACIENTES


if _hay_subset_local():
    print(f"Subset ya disponible en {BASE_DIR} -> no se descarga nada.")

elif os.path.exists(RESPALDO_ZIP):
    print("Respaldo encontrado en Drive -> copiando y extrayendo localmente...")
    local_zip = "/content/brats_subset_local.zip"
    shutil.copy(RESPALDO_ZIP, local_zip)
    with zipfile.ZipFile(local_zip, "r") as z:
        z.extractall(OUT_DIR)
    os.remove(local_zip)
    print("Listo.")

else:
    print("No hay respaldo en Drive -> descargando de Kaggle (solo ocurre una vez)...")
    !kaggle datasets download -d {KAGGLE_DATASET} -p /content/brats

    zip_path = f"/content/brats/{KAGGLE_DATASET.split('/')[-1]}.zip"
    with zipfile.ZipFile(zip_path, "r") as z:
        nombres = z.namelist()
        pacientes = sorted(set(n.split("/")[1] for n in nombres if "/" in n))
        pacientes_set = set(pacientes[:N_PACIENTES])
        for n in nombres:
            partes = n.split("/")
            if len(partes) > 1 and partes[1] in pacientes_set:
                z.extract(n, OUT_DIR)
    os.remove(zip_path)       # libera espacio del zip completo (~147 GB)
    print(f"Pacientes extraídos: {N_PACIENTES}")

    print("Creando respaldo comprimido en Drive (tarda una vez; ahorra las siguientes)...")
    local_zip = "/content/brats_subset_local.zip"
    !cd {OUT_DIR} && zip -r -q "{local_zip}" .
    shutil.copy(local_zip, RESPALDO_ZIP)
    os.remove(local_zip)
    print("Respaldo guardado en:", RESPALDO_ZIP)

!df -h /content

In [ ]:
# ──────────────────────────────────────────────────────────────────
# [3.3] Utilidades de datos
# ──────────────────────────────────────────────────────────────────
# Qué hace : Funciones auxiliares: localizar las rutas NIfTI de un paciente,
#            normalizar intensidades a [-1, 1] y seleccionar cortes axiales
#            con cerebro. get_paciente_paths acepta tanto archivos
#            .nii/.nii.gz como carpetas que contienen el .nii.
# Produce  : get_paciente_paths, normalizar, cortes_validos
# ──────────────────────────────────────────────────────────────────
def _resolver(ruta):
    """Si `ruta` es una carpeta, devuelve el primer .nii* que contiene."""
    if ruta is None:
        return None
    if os.path.isdir(ruta):
        dentro = glob.glob(os.path.join(ruta, "*.nii*"))
        return dentro[0] if dentro else None
    return ruta


def get_paciente_paths(paciente_dir):
    """Devuelve un dict con las rutas a seg, t1c, t1n, t2f y t2w de un paciente
    (None si falta alguna)."""
    paths = {}
    for clave, patron in [("seg", "*-seg.nii*"), ("t1c", "*-t1c.nii*"), ("t1n", "*-t1n.nii*"),
                          ("t2f", "*-t2f.nii*"), ("t2w", "*-t2w.nii*")]:
        cand = glob.glob(os.path.join(paciente_dir, patron))
        paths[clave] = _resolver(cand[0]) if cand else None
    return paths


def normalizar(img):
    """Recorta en el percentil 99 y escala a [-1, 1]."""
    p99 = np.percentile(img, 99)
    img = np.clip(img, 0, p99)
    img = img / (p99 + 1e-8)
    img = img * 2 - 1
    return img.astype(np.float32)


def cortes_validos(volumen_t1n, umbral_pixeles=1000):
    """Índices z de los cortes axiales con más de `umbral_pixeles` píxeles > 0."""
    return [z for z in range(volumen_t1n.shape[2])
            if np.sum(volumen_t1n[:, :, z] > 0) > umbral_pixeles]

In [ ]:
# ──────────────────────────────────────────────────────────────────
# [3.4] Dataset BraTS (T1n → T1ce + máscara tumoral)
# ──────────────────────────────────────────────────────────────────
# Qué hace : Clase Dataset que devuelve (T1n, T1ce, máscara) por corte axial.
#            Usa los MAX_CORTES cortes centrales de cada paciente.
#            Augmentation conservador: solo flip horizontal.
# Produce  : BraTSPix2PixDatasetAugV2
# ──────────────────────────────────────────────────────────────────
class BraTSPix2PixDatasetAugV2(Dataset):
    def __init__(self, pacientes_dirs, max_cortes_por_paciente=20, size=256, augment=True):
        self.pares = []          # (ruta_t1n, ruta_t1c, ruta_seg, z)
        self.size = size
        self.augment = augment
        for pdir in pacientes_dirs:
            paths = get_paciente_paths(pdir)
            if paths['t1n'] is None or paths['t1c'] is None or paths['seg'] is None:
                continue
            t1n_img = nib.load(paths['t1n']).get_fdata()
            validos = cortes_validos(t1n_img)
            centro = len(validos) // 2
            rango = validos[max(0, centro - max_cortes_por_paciente // 2):
                            centro + max_cortes_por_paciente // 2]
            for z in rango:
                self.pares.append((paths['t1n'], paths['t1c'], paths['seg'], z))

    def __len__(self):
        return len(self.pares)

    def __getitem__(self, idx):
        ruta_t1n, ruta_t1c, ruta_seg, z = self.pares[idx]
        t1n = nib.load(ruta_t1n).get_fdata()[:, :, z]
        t1c = nib.load(ruta_t1c).get_fdata()[:, :, z]
        seg = nib.load(ruta_seg).get_fdata()[:, :, z]

        t1n = normalizar(t1n)
        t1c = normalizar(t1c)
        mascara_tumor = (seg > 0).astype(np.float32)

        t1n = torch.from_numpy(t1n).unsqueeze(0)
        t1c = torch.from_numpy(t1c).unsqueeze(0)
        mascara_tumor = torch.from_numpy(mascara_tumor).unsqueeze(0)

        t1n = F.interpolate(t1n.unsqueeze(0), size=(self.size, self.size), mode='bilinear', align_corners=False).squeeze(0)
        t1c = F.interpolate(t1c.unsqueeze(0), size=(self.size, self.size), mode='bilinear', align_corners=False).squeeze(0)
        mascara_tumor = F.interpolate(mascara_tumor.unsqueeze(0), size=(self.size, self.size), mode='nearest').squeeze(0)

        # Solo flip horizontal (el cerebro es aprox. simétrico); sin rotaciones,
        # que son anatómicamente incorrectas en MRI axial.
        if self.augment and random.random() > 0.5:
            t1n = torch.flip(t1n, dims=[2])
            t1c = torch.flip(t1c, dims=[2])
            mascara_tumor = torch.flip(mascara_tumor, dims=[2])

        return t1n, t1c, mascara_tumor

In [ ]:
# ──────────────────────────────────────────────────────────────────
# [3.5] Split por paciente y DataLoaders
# ──────────────────────────────────────────────────────────────────
# Qué hace : Divide los pacientes en train/val/test (70/15/15, split FIJO) y
#            crea los datasets y loaders de val y test. El train_loader se
#            recrea dentro de cada corrida para que el shuffle dependa de la
#            semilla. El test_loader solo se usa en evaluar_en_test().
# Produce  : train_dirs, val_dirs, test_dirs, train_dataset, val_dataset,
#            test_dataset, val_loader, test_loader
# ──────────────────────────────────────────────────────────────────
pacientes_dirs = sorted(glob.glob(os.path.join(BASE_DIR, "BraTS-GLI-*")))
print("Total pacientes en el subset:", len(pacientes_dirs))


def crear_split_pacientes(lista_pacientes, seed_split=SEED_SPLIT):
    """Split fijo por paciente. Es el mismo en todas las corridas: lo que varía
    entre semillas es la inicialización y el shuffle, no qué pacientes caen en
    cada partición."""
    train_dirs, temp_dirs = train_test_split(lista_pacientes, test_size=FRAC_TEMP, random_state=seed_split)
    val_dirs, test_dirs = train_test_split(temp_dirs, test_size=0.50, random_state=seed_split)
    print(f"Split fijo (seed={seed_split}): train={len(train_dirs)}, val={len(val_dirs)}, test={len(test_dirs)}")
    return train_dirs, val_dirs, test_dirs


train_dirs, val_dirs, test_dirs = crear_split_pacientes(pacientes_dirs)

# Train con augmentation; val y test sin ella.
train_dataset = BraTSPix2PixDatasetAugV2(train_dirs, MAX_CORTES, IMG_SIZE, augment=True)
val_dataset   = BraTSPix2PixDatasetAugV2(val_dirs,   MAX_CORTES, IMG_SIZE, augment=False)
test_dataset  = BraTSPix2PixDatasetAugV2(test_dirs,  MAX_CORTES, IMG_SIZE, augment=False)

if MODO_PRUEBA:   # dataset reducido para la prueba rápida
    train_dataset = Subset(train_dataset, range(min(40, len(train_dataset))))
    val_dataset   = Subset(val_dataset,   range(min(16, len(val_dataset))))
    test_dataset  = Subset(test_dataset,  range(min(16, len(test_dataset))))

print("Cortes -> train:", len(train_dataset), "| val:", len(val_dataset), "| test:", len(test_dataset))

val_loader  = DataLoader(val_dataset,  batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=NUM_WORKERS, worker_init_fn=worker_init_fn)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=NUM_WORKERS, worker_init_fn=worker_init_fn)

In [ ]:
# ──────────────────────────────────────────────────────────────────
# [3.6] Verificación de integridad del split
# ──────────────────────────────────────────────────────────────────
# Qué hace : Comprueba que los pacientes tengan los archivos necesarios (t1n,
#            t1c, seg) y que NO haya pacientes compartidos entre train, val y
#            test (sin fuga de datos).
# ──────────────────────────────────────────────────────────────────
def pid(p):
    """ID numérico del paciente (p. ej. '00012') a partir de su carpeta."""
    return re.search(r"BraTS-GLI-(\d{5})-", os.path.basename(p)).group(1)


def completos(dirs):
    """Pacientes con t1n, t1c y seg disponibles."""
    return [p for p in dirs
            if all(get_paciente_paths(p)[k] is not None for k in ["t1n", "t1c", "seg"])]


splits = {"train": train_dirs, "val": val_dirs, "test": test_dirs}
usados = {k: completos(v) for k, v in splits.items()}
ids = {k: {pid(p) for p in v} for k, v in usados.items()}

for nombre, d in splits.items():
    print(f"{nombre:5s}: {len(d)} pacientes | {len(usados[nombre])} con archivos completos "
          f"| cortes esperados <= {MAX_CORTES * len(usados[nombre])}")

print("\nPacientes únicos usados:", len(set().union(*ids.values())))
for a, b in [("train", "val"), ("train", "test"), ("val", "test")]:
    compartidos = sorted(ids[a] & ids[b])
    print(f"{a} ∩ {b}: {len(compartidos)} pacientes compartidos {compartidos if compartidos else ''}")
    assert not compartidos, f"Fuga de datos entre {a} y {b}"

incompletos = [p for p in test_dirs if p not in usados["test"]]
if incompletos:
    print("\nEjemplo de paciente incompleto en test:", os.path.basename(incompletos[0]))
    print(get_paciente_paths(incompletos[0]))

In [ ]:
# ──────────────────────────────────────────────────────────────────
# [3.7] Estadísticas del conjunto de test
# ──────────────────────────────────────────────────────────────────
# Qué hace : Cuenta cortes por paciente y cortes de test que contienen tumor.
#            Recorre todo el test (puede tardar un par de minutos); se omite
#            en MODO_PRUEBA.
# ──────────────────────────────────────────────────────────────────
if MODO_PRUEBA:
    print("MODO_PRUEBA: se omite (el test es un Subset reducido).")
else:
    por_paciente = pd.Series([p[0] for p in test_dataset.pares]).value_counts()
    print("Cortes por paciente en test:\n", por_paciente.describe())
    n_con_tumor = sum(int(test_dataset[i][2].sum() > 0) for i in range(len(test_dataset)))
    print(f"\nCortes de test con tumor: {n_con_tumor} de {len(test_dataset)}")

## 4. Modelos
Generador U-Net y tres discriminadores:

| Clase | Usada en | Notas |
|---|---|---|
| `DiscriminadorPatchGAN` | `baseline` | PatchGAN estándar, sin spectral norm |
| `DiscriminadorPatchGANv2` | `warmup_ema` | PatchGAN con `spectral_norm` en cada convolución |
| `DiscriminadorTumorAware` | variantes `tumor_aware_*` | `spectral_norm` + máscara tumoral como 3.er canal |

In [ ]:
# ──────────────────────────────────────────────────────────────────
# [4.1] Bloques del generador
# ──────────────────────────────────────────────────────────────────
# Qué hace : Bloques de codificación (Conv stride 2) y de decodificación
#            (Upsample + Conv + concatenación con la skip connection) de la
#            U-Net.
# Produce  : BloqueDown, BloqueUpMejorado
# ──────────────────────────────────────────────────────────────────
class BloqueDown(nn.Module):
    def __init__(self, in_ch, out_ch, normalizar=True):
        super().__init__()
        capas = [nn.Conv2d(in_ch, out_ch, 4, stride=2, padding=1)]
        if normalizar:
            capas.append(nn.BatchNorm2d(out_ch))
        capas.append(nn.LeakyReLU(0.2))
        self.bloque = nn.Sequential(*capas)

    def forward(self, x):
        return self.bloque(x)


class BloqueUpMejorado(nn.Module):
    def __init__(self, in_ch, out_ch, dropout=False):
        super().__init__()
        capas = [
            nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False),
            nn.Conv2d(in_ch, out_ch, 3, stride=1, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU()
        ]
        if dropout:
            capas.append(nn.Dropout(0.5))
        self.bloque = nn.Sequential(*capas)

    def forward(self, x, skip):
        x = self.bloque(x)
        return torch.cat([x, skip], dim=1)

In [ ]:
# ──────────────────────────────────────────────────────────────────
# [4.2] Generador U-Net
# ──────────────────────────────────────────────────────────────────
# Qué hace : U-Net de 6 niveles (256×256 → 4×4) con skip connections; salida
#            Tanh en [-1, 1].
# Produce  : GeneradorUNetV2
# ──────────────────────────────────────────────────────────────────
class GeneradorUNetV2(nn.Module):
    def __init__(self, in_ch=1, out_ch=1):
        super().__init__()
        self.down1 = BloqueDown(in_ch, 64, normalizar=False)
        self.down2 = BloqueDown(64, 128)
        self.down3 = BloqueDown(128, 256)
        self.down4 = BloqueDown(256, 512)
        self.down5 = BloqueDown(512, 512)
        self.down6 = BloqueDown(512, 512, normalizar=False)

        self.up1 = BloqueUpMejorado(512, 512, dropout=True)
        self.up2 = BloqueUpMejorado(1024, 512, dropout=True)
        self.up3 = BloqueUpMejorado(1024, 256)
        self.up4 = BloqueUpMejorado(512, 128)
        self.up5 = BloqueUpMejorado(256, 64)

        self.final = nn.Sequential(
            nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False),
            nn.Conv2d(128, out_ch, 3, stride=1, padding=1),
            nn.Tanh()
        )

    def forward(self, x):
        d1 = self.down1(x); d2 = self.down2(d1); d3 = self.down3(d2)
        d4 = self.down4(d3); d5 = self.down5(d4); d6 = self.down6(d5)
        u1 = self.up1(d6, d5); u2 = self.up2(u1, d4); u3 = self.up3(u2, d3)
        u4 = self.up4(u3, d2); u5 = self.up5(u4, d1)
        return self.final(u5)

In [ ]:
# ──────────────────────────────────────────────────────────────────
# [4.3] Discriminadores PatchGAN (sin y con spectral norm)
# ──────────────────────────────────────────────────────────────────
# Qué hace : DiscriminadorPatchGAN: Pix2Pix original, solo para la variante
#            baseline. DiscriminadorPatchGANv2: añade spectral_norm en cada
#            convolución, usado desde warmup_ema.
# Produce  : DiscriminadorPatchGAN, DiscriminadorPatchGANv2
# ──────────────────────────────────────────────────────────────────
class DiscriminadorPatchGAN(nn.Module):
    """PatchGAN 'vanilla' SIN spectral_norm. Solo en la variante `baseline`
    (replica el Pix2Pix original, sin estabilización adicional)."""
    def __init__(self, in_ch=2):
        super().__init__()
        self.modelo = nn.Sequential(
            nn.Conv2d(in_ch, 64, 4, stride=2, padding=1), nn.LeakyReLU(0.2),
            nn.Conv2d(64, 128, 4, stride=2, padding=1), nn.BatchNorm2d(128), nn.LeakyReLU(0.2),
            nn.Conv2d(128, 256, 4, stride=2, padding=1), nn.BatchNorm2d(256), nn.LeakyReLU(0.2),
            nn.Conv2d(256, 512, 4, stride=1, padding=1), nn.BatchNorm2d(512), nn.LeakyReLU(0.2),
            nn.Conv2d(512, 1, 4, stride=1, padding=1)
        )

    def forward(self, img_input, img_target):
        x = torch.cat([img_input, img_target], dim=1)
        return self.modelo(x)


class DiscriminadorPatchGANv2(nn.Module):
    """PatchGAN CON spectral_norm en cada capa convolucional. Se usa desde la
    variante `warmup_ema` (en `tumor_aware_*` se reemplaza por
    DiscriminadorTumorAware)."""
    def __init__(self, in_ch=2):
        super().__init__()
        self.modelo = nn.Sequential(
            nn.utils.spectral_norm(nn.Conv2d(in_ch, 64, 4, stride=2, padding=1)),
            nn.LeakyReLU(0.2),
            nn.utils.spectral_norm(nn.Conv2d(64, 128, 4, stride=2, padding=1)),
            nn.BatchNorm2d(128), nn.LeakyReLU(0.2),
            nn.utils.spectral_norm(nn.Conv2d(128, 256, 4, stride=2, padding=1)),
            nn.BatchNorm2d(256), nn.LeakyReLU(0.2),
            nn.utils.spectral_norm(nn.Conv2d(256, 512, 4, stride=1, padding=1)),
            nn.BatchNorm2d(512), nn.LeakyReLU(0.2),
            nn.utils.spectral_norm(nn.Conv2d(512, 1, 4, stride=1, padding=1))
        )

    def forward(self, img_input, img_target):
        return self.modelo(torch.cat([img_input, img_target], dim=1))

In [ ]:
# ──────────────────────────────────────────────────────────────────
# [4.4] Discriminador tumor-aware
# ──────────────────────────────────────────────────────────────────
# Qué hace : PatchGAN con spectral_norm que recibe además la máscara tumoral
#            como tercer canal de entrada (T1n + T1ce real/generado +
#            máscara).
# Produce  : DiscriminadorTumorAware
# ──────────────────────────────────────────────────────────────────
class DiscriminadorTumorAware(nn.Module):
    """Recibe la máscara tumoral como tercer canal, además de input y target.
    También usa spectral_norm (igual que DiscriminadorPatchGANv2)."""
    def __init__(self, in_ch=3):          # t1n + t1c (real o falso) + máscara
        super().__init__()
        self.modelo = nn.Sequential(
            nn.utils.spectral_norm(nn.Conv2d(in_ch, 64, 4, stride=2, padding=1)),
            nn.LeakyReLU(0.2),
            nn.utils.spectral_norm(nn.Conv2d(64, 128, 4, stride=2, padding=1)),
            nn.BatchNorm2d(128), nn.LeakyReLU(0.2),
            nn.utils.spectral_norm(nn.Conv2d(128, 256, 4, stride=2, padding=1)),
            nn.BatchNorm2d(256), nn.LeakyReLU(0.2),
            nn.utils.spectral_norm(nn.Conv2d(256, 512, 4, stride=1, padding=1)),
            nn.BatchNorm2d(512), nn.LeakyReLU(0.2),
            nn.utils.spectral_norm(nn.Conv2d(512, 1, 4, stride=1, padding=1))
        )

    def forward(self, img_input, img_target, mascara):
        x = torch.cat([img_input, img_target, mascara], dim=1)
        return self.modelo(x)

## 5. Funciones de pérdida
Pérdidas base (GAN y L1) y los términos adicionales del generador: SSIM diferenciable, pérdida de frecuencia (estilo BraSyn) y pérdida perceptual VGG.

In [ ]:
# ──────────────────────────────────────────────────────────────────
# [5.1] Pérdidas base
# ──────────────────────────────────────────────────────────────────
# Qué hace : Criterios GAN (BCE con logits, versión promediada y versión por
#            elemento para ponderar parches) y L1.
# Produce  : criterio_gan, criterio_gan_elemwise, criterio_l1
# ──────────────────────────────────────────────────────────────────
criterio_gan = nn.BCEWithLogitsLoss()
criterio_gan_elemwise = nn.BCEWithLogitsLoss(reduction='none')
criterio_l1 = nn.L1Loss()

In [ ]:
# ──────────────────────────────────────────────────────────────────
# [5.2] Pérdida SSIM diferenciable
# ──────────────────────────────────────────────────────────────────
# Qué hace : SSIM con ventana gaussiana (Wang et al., 2004) como término
#            ADICIONAL de entrenamiento del generador, siguiendo la práctica
#            de FermatSyn (arXiv:2505.07687) y del ganador de BraSyn 2023
#            (Baltruschat et al., 2024): L1 + SSIM + adversarial.
# Produce  : SSIMLoss
# ──────────────────────────────────────────────────────────────────
class SSIMLoss(nn.Module):
    def __init__(self, kernel_size=11, sigma=1.5, data_range=1.0):
        super().__init__()
        self.data_range = data_range
        coords = torch.arange(kernel_size, dtype=torch.float32) - kernel_size // 2
        g = torch.exp(-(coords ** 2) / (2 * sigma ** 2))
        g = (g / g.sum()).unsqueeze(0)
        kernel_2d = g.T @ g
        self.register_buffer("kernel", kernel_2d.unsqueeze(0).unsqueeze(0))
        self.pad = kernel_size // 2

    def forward(self, generado, real):
        # de [-1, 1] a [0, 1] (mismo criterio que el SSIM/PSNR de evaluación)
        img1 = (generado + 1) / 2
        img2 = (real + 1) / 2

        C1 = (0.01 * self.data_range) ** 2
        C2 = (0.03 * self.data_range) ** 2

        mu1 = F.conv2d(img1, self.kernel, padding=self.pad)
        mu2 = F.conv2d(img2, self.kernel, padding=self.pad)
        mu1_sq, mu2_sq, mu1_mu2 = mu1 ** 2, mu2 ** 2, mu1 * mu2

        sigma1_sq = F.conv2d(img1 * img1, self.kernel, padding=self.pad) - mu1_sq
        sigma2_sq = F.conv2d(img2 * img2, self.kernel, padding=self.pad) - mu2_sq
        sigma12 = F.conv2d(img1 * img2, self.kernel, padding=self.pad) - mu1_mu2

        ssim_map = ((2 * mu1_mu2 + C1) * (2 * sigma12 + C2)) / (
            (mu1_sq + mu2_sq + C1) * (sigma1_sq + sigma2_sq + C2)
        )
        return 1 - ssim_map.mean()

In [ ]:
# ──────────────────────────────────────────────────────────────────
# [5.3] Pérdida de frecuencia (estilo BraSyn)
# ──────────────────────────────────────────────────────────────────
# Qué hace : Separa el espectro de magnitud de Fourier en baja frecuencia
#            (círculo central de radio r=21, como en Baltruschat et al.,
#            2024, para imágenes 256×256) y alta frecuencia (complemento), y
#            penaliza cada banda por separado con L1.
# Produce  : FrequencyLoss
# ──────────────────────────────────────────────────────────────────
class FrequencyLoss(nn.Module):
    """Usa torch.fft.fft2(..., norm='ortho'). Sin esa normalización, la magnitud
    del espectro es ~sqrt(H*W) veces mayor que los valores espaciales (~256x para
    256×256), de modo que esta pérdida dominaría sobre L1 y la adversarial y
    desestabilizaría el entrenamiento."""
    def __init__(self, radius=21, img_size=256):
        super().__init__()
        yy, xx = torch.meshgrid(
            torch.arange(img_size) - img_size // 2,
            torch.arange(img_size) - img_size // 2,
            indexing="ij",
        )
        dist = torch.sqrt((xx ** 2 + yy ** 2).float())
        mask_low = (dist <= radius).float()
        self.register_buffer("mask_low", mask_low.unsqueeze(0).unsqueeze(0))
        self.register_buffer("mask_high", 1 - self.mask_low)

    def forward(self, generado, real):
        F_gen = torch.fft.fftshift(torch.fft.fft2(generado, norm="ortho"), dim=(-2, -1))
        F_real = torch.fft.fftshift(torch.fft.fft2(real, norm="ortho"), dim=(-2, -1))
        mag_gen, mag_real = torch.abs(F_gen), torch.abs(F_real)

        loss_low = F.l1_loss(mag_gen * self.mask_low, mag_real * self.mask_low)
        loss_high = F.l1_loss(mag_gen * self.mask_high, mag_real * self.mask_high)
        return loss_low + loss_high

In [ ]:
# ──────────────────────────────────────────────────────────────────
# [5.4] Pérdida perceptual VGG (opcional)
# ──────────────────────────────────────────────────────────────────
# Qué hace : VGG-16 preentrenada en ImageNet y congelada. Se incluye para
#            poder reproducir la variante con VGG (hallazgo negativo por
#            mismatch de dominio natural vs. MRI); NO se usa en las variantes
#            finales.
# Produce  : VGGPerceptualLoss
# ──────────────────────────────────────────────────────────────────
class VGGPerceptualLoss(nn.Module):
    def __init__(self, layers=(3, 8, 15, 22)):
        super().__init__()
        vgg = tv_models.vgg16(weights=tv_models.VGG16_Weights.IMAGENET1K_V1).features
        self.layers = layers
        self.blocks = nn.ModuleList()
        prev = 0
        for l in layers:
            self.blocks.append(vgg[prev:l + 1])
            prev = l + 1
        for p in self.parameters():
            p.requires_grad_(False)
        self.eval()

    def forward(self, generado, real):
        # 1 canal -> 3 canales (VGG espera RGB); [-1, 1] -> [0, 1]
        gen_rgb = ((generado + 1) / 2).repeat(1, 3, 1, 1)
        real_rgb = ((real + 1) / 2).repeat(1, 3, 1, 1)
        loss = 0.0
        x_g, x_r = gen_rgb, real_rgb
        for block in self.blocks:
            x_g = block(x_g)
            x_r = block(x_r)
            loss = loss + F.l1_loss(x_g, x_r)
        return loss

## 6. Métricas de evaluación
SSIM y PSNR **globales** (imagen completa) y **tumorales** (solo dentro de la máscara), calculados en una única pasada por el loader.

In [ ]:
# ──────────────────────────────────────────────────────────────────
# [6.1] Evaluación combinada (global + tumoral)
# ──────────────────────────────────────────────────────────────────
# Qué hace : Calcula SSIM/PSNR global y tumoral sobre un loader. Usa skimage
#            con data_range=1.0 y reescala de [-1,1] a [0,1]. SSIM tumoral =
#            mapa SSIM (full=True) promediado dentro de la máscara; PSNR
#            tumoral = calculado manualmente por MSE dentro de la máscara.
# Produce  : evaluar_completo
# ──────────────────────────────────────────────────────────────────
@torch.no_grad()
def evaluar_completo(gen, loader, device):
    gen.eval()
    ssim_g_tot, psnr_g_tot, n_g = 0, 0, 0
    ssim_t_tot, psnr_t_tot, n_t = 0, 0, 0

    for t1n, t1c, mascara in loader:
        t1n, t1c, mascara = t1n.to(device), t1c.to(device), mascara.to(device)
        t1c_falso = gen(t1n)

        for i in range(t1n.shape[0]):
            real = ((t1c[i, 0].cpu().numpy() + 1) / 2)
            falso = ((t1c_falso[i, 0].cpu().numpy() + 1) / 2)
            m = mascara[i, 0].cpu().numpy() > 0

            ssim_global, mapa = ssim(real, falso, data_range=1.0, full=True)
            psnr_global = psnr(real, falso, data_range=1.0)
            ssim_g_tot += ssim_global
            psnr_g_tot += psnr_global
            n_g += 1

            if m.sum() > 0:      # métricas tumorales solo en cortes con tumor
                ssim_t_tot += mapa[m].mean()
                mse_tumor = np.mean((real[m] - falso[m]) ** 2)
                psnr_t_tot += 10 * np.log10(1.0 / (mse_tumor + 1e-8))
                n_t += 1

    return {
        "ssim_global": ssim_g_tot / n_g,
        "psnr_global": psnr_g_tot / n_g,
        "ssim_tumor": ssim_t_tot / n_t if n_t > 0 else None,
        "psnr_tumor": psnr_t_tot / n_t if n_t > 0 else None,
    }

In [ ]:
# ──────────────────────────────────────────────────────────────────
# [6.2] Evaluación final en TEST
# ──────────────────────────────────────────────────────────────────
# Qué hace : Carga el mejor checkpoint de una corrida y lo evalúa sobre el
#            test_loader. Se llama UNA sola vez por corrida; el test nunca
#            interviene en el entrenamiento ni en la selección de
#            checkpoints.
# Produce  : evaluar_en_test
# ──────────────────────────────────────────────────────────────────
def evaluar_en_test(ckpt_path):
    gen = GeneradorUNetV2().to(device)
    gen.load_state_dict(torch.load(ckpt_path, map_location=device, weights_only=False))
    return evaluar_completo(gen, test_loader, device)

## 7. Entrenamiento
Warm-up, EMA, los dos tipos de época de entrenamiento (discriminador estándar y tumor-aware) y la función unificada `entrenar_variante()`, **con reanudación automática** desde el último estado guardado.

In [ ]:
# ──────────────────────────────────────────────────────────────────
# [7.1] Warm-up del generador
# ──────────────────────────────────────────────────────────────────
# Qué hace : Entrena solo el generador con L1 ponderado por tumor (sin
#            discriminador) durante unas pocas épocas, antes del
#            entrenamiento adversarial.
# Produce  : warmup_generador
# ──────────────────────────────────────────────────────────────────
def warmup_generador(gen, loader, opt_gen, device, lambda_l1, peso_tumor=5.0, n_epocas=5):
    gen.train()
    for epoca in range(n_epocas):
        perdida_total = 0
        for t1n, t1c, mascara in loader:
            t1n, t1c, mascara = t1n.to(device), t1c.to(device), mascara.to(device)
            t1c_falso = gen(t1n)
            mapa_pesos = 1.0 + mascara * (peso_tumor - 1.0)
            error_l1 = torch.abs(t1c_falso - t1c)
            loss = (error_l1 * mapa_pesos).mean() * lambda_l1
            opt_gen.zero_grad()
            loss.backward()
            opt_gen.step()
            perdida_total += loss.item()
        print(f"  Warm-up época {epoca + 1}/{n_epocas} | Loss L1: {perdida_total / len(loader):.3f}")

In [ ]:
# ──────────────────────────────────────────────────────────────────
# [7.2] EMA (media móvil exponencial)
# ──────────────────────────────────────────────────────────────────
# Qué hace : Mantiene una copia del generador con pesos promediados
#            exponencialmente; es la que se evalúa y se guarda como
#            checkpoint. decay=0.97 en todas las corridas (0.999 es demasiado
#            lento para converger en solo 40 épocas adversariales).
# Produce  : EMA
# ──────────────────────────────────────────────────────────────────
class EMA:
    def __init__(self, modelo, decay=0.97):
        self.decay = decay
        self.modelo_ema = copy.deepcopy(modelo)
        for p in self.modelo_ema.parameters():
            p.requires_grad_(False)

    def update(self, modelo):
        with torch.no_grad():
            for p_ema, p in zip(self.modelo_ema.parameters(), modelo.parameters()):
                p_ema.mul_(self.decay).add_(p.detach(), alpha=1 - self.decay)

    def eval_modelo(self):
        return self.modelo_ema

In [ ]:
# ──────────────────────────────────────────────────────────────────
# [7.3] Época de entrenamiento con discriminador estándar
# ──────────────────────────────────────────────────────────────────
# Qué hace : Una época adversarial con L1 ponderado por tumor y discriminador
#            SIN máscara. Usada por las variantes baseline y warmup_ema.
# Produce  : entrenar_epoca_ponderada
# ──────────────────────────────────────────────────────────────────
def entrenar_epoca_ponderada(gen, disc, loader, opt_gen, opt_disc, device, lambda_l1, peso_tumor=5.0):
    gen.train(); disc.train()
    perdida_gen_total = 0; perdida_disc_total = 0

    for t1n, t1c, mascara in loader:
        t1n, t1c, mascara = t1n.to(device), t1c.to(device), mascara.to(device)

        # --- Discriminador ---
        with torch.no_grad():
            t1c_falso = gen(t1n)
        pred_real = disc(t1n, t1c)
        pred_falso = disc(t1n, t1c_falso)
        loss_d = (criterio_gan(pred_real, torch.ones_like(pred_real)) +
                  criterio_gan(pred_falso, torch.zeros_like(pred_falso))) / 2
        opt_disc.zero_grad(); loss_d.backward(); opt_disc.step()

        # --- Generador ---
        t1c_falso = gen(t1n)
        pred_falso = disc(t1n, t1c_falso)
        loss_g_gan = criterio_gan(pred_falso, torch.ones_like(pred_falso))

        mapa_pesos = 1.0 + mascara * (peso_tumor - 1.0)
        error_l1 = torch.abs(t1c_falso - t1c)
        loss_g_l1 = (error_l1 * mapa_pesos).mean() * lambda_l1
        loss_g = loss_g_gan + loss_g_l1

        opt_gen.zero_grad(); loss_g.backward(); opt_gen.step()

        perdida_gen_total += loss_g.item()
        perdida_disc_total += loss_d.item()

    return perdida_gen_total / len(loader), perdida_disc_total / len(loader)

In [ ]:
# ──────────────────────────────────────────────────────────────────
# [7.4] Época de entrenamiento con discriminador tumor-aware
# ──────────────────────────────────────────────────────────────────
# Qué hace : Una época adversarial donde el discriminador recibe la máscara y
#            la pérdida GAN pondera más los parches que caen sobre el tumor
#            (peso_disc_tumor). Admite términos opcionales SSIM, frecuencia y
#            VGG en el generador.
# Produce  : entrenar_epoca_disc_tumoraware
# ──────────────────────────────────────────────────────────────────
def entrenar_epoca_disc_tumoraware(gen, disc, loader, opt_gen, opt_disc, device,
                                   lambda_l1, peso_tumor=5.0, peso_disc_tumor=3.0,
                                   ssim_loss_fn=None, freq_loss_fn=None, vgg_loss_fn=None,
                                   lambda_ssim=0.0, lambda_freq=0.0, lambda_vgg=0.0):
    gen.train(); disc.train()
    perdida_gen_total = 0; perdida_disc_total = 0

    for t1n, t1c, mascara in loader:
        t1n, t1c, mascara = t1n.to(device), t1c.to(device), mascara.to(device)

        # --- Discriminador ---
        with torch.no_grad():
            t1c_falso = gen(t1n)

        pred_real = disc(t1n, t1c, mascara)
        pred_falso = disc(t1n, t1c_falso, mascara)

        # Peso por parche: 1 fuera del tumor, `peso_disc_tumor` dentro
        mascara_parches = F.adaptive_avg_pool2d(mascara, pred_real.shape[-2:])
        mapa_pesos = 1.0 + mascara_parches * (peso_disc_tumor - 1.0)

        loss_d_real = (criterio_gan_elemwise(pred_real, torch.ones_like(pred_real)) * mapa_pesos).mean()
        loss_d_falso = (criterio_gan_elemwise(pred_falso, torch.zeros_like(pred_falso)) * mapa_pesos).mean()
        loss_d = (loss_d_real + loss_d_falso) / 2

        opt_disc.zero_grad(); loss_d.backward(); opt_disc.step()

        # --- Generador ---
        t1c_falso = gen(t1n)
        pred_falso = disc(t1n, t1c_falso, mascara)
        loss_g_gan = (criterio_gan_elemwise(pred_falso, torch.ones_like(pred_falso)) * mapa_pesos).mean()

        mapa_pesos_l1 = 1.0 + mascara * (peso_tumor - 1.0)
        error_l1 = torch.abs(t1c_falso - t1c)
        loss_g_l1 = (error_l1 * mapa_pesos_l1).mean() * lambda_l1

        loss_g = loss_g_gan + loss_g_l1

        if ssim_loss_fn is not None:
            loss_g = loss_g + lambda_ssim * ssim_loss_fn(t1c_falso, t1c)
        if freq_loss_fn is not None:
            loss_g = loss_g + lambda_freq * freq_loss_fn(t1c_falso, t1c)
        if vgg_loss_fn is not None:
            loss_g = loss_g + lambda_vgg * vgg_loss_fn(t1c_falso, t1c)

        opt_gen.zero_grad(); loss_g.backward(); opt_gen.step()

        perdida_gen_total += loss_g.item()
        perdida_disc_total += loss_d.item()

    return perdida_gen_total / len(loader), perdida_disc_total / len(loader)

In [ ]:
# ──────────────────────────────────────────────────────────────────
# [7.5] entrenar_variante(): entrenamiento unificado con reanudación
# ──────────────────────────────────────────────────────────────────
# Qué hace : Entrena UNA variante con UNA semilla: construye modelos según la
#            config, hace warm-up, entrena N_EPOCAS, evalúa en validación
#            cada 5 épocas (y en la 1.ª), guarda el mejor checkpoint por SSIM
#            tumoral, y guarda el estado completo cada época para poder
#            reanudar. Devuelve las métricas de validación del mejor
#            checkpoint.
# Produce  : entrenar_variante
# ──────────────────────────────────────────────────────────────────
def entrenar_variante(nombre_variante, config, seed, checkpoint_dir=None):
    seed_everything(seed)
    if checkpoint_dir is None:
        checkpoint_dir = CHECKPOINT_DIR
    os.makedirs(checkpoint_dir, exist_ok=True)

    estado_path = os.path.join(checkpoint_dir, f"{nombre_variante}_seed{seed}_estado.pth")
    mejor_ckpt_path = os.path.join(checkpoint_dir, f"{nombre_variante}_seed{seed}_best.pth")
    historial_path = os.path.join(checkpoint_dir, f"{nombre_variante}_seed{seed}_historial.csv")

    # El train_loader se recrea aquí para que el shuffle dependa de la semilla.
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                              num_workers=NUM_WORKERS, worker_init_fn=worker_init_fn)

    # --- Modelos ---
    gen = GeneradorUNetV2().to(device)
    if config["usar_disc_tumor_aware"]:
        disc = DiscriminadorTumorAware().to(device)
    elif config["usar_spectral_norm"]:
        disc = DiscriminadorPatchGANv2().to(device)
    else:
        disc = DiscriminadorPatchGAN().to(device)

    opt_gen = torch.optim.Adam(gen.parameters(), lr=LR_GEN, betas=(0.5, 0.999))
    opt_disc = torch.optim.Adam(disc.parameters(), lr=LR_DISC, betas=(0.5, 0.999))

    # --- Pérdidas opcionales y EMA ---
    ssim_loss_fn = SSIMLoss().to(device) if config["usar_ssim_loss"] else None
    freq_loss_fn = FrequencyLoss().to(device) if config["usar_freq_loss"] else None
    vgg_loss_fn = VGGPerceptualLoss().to(device) if config.get("usar_vgg_loss") else None
    ema = EMA(gen, decay=EMA_DECAY) if config["usar_ema"] else None

    # Peso de parches tumorales en D: el de la variante, o el valor por defecto
    peso_disc_tumor = config.get("peso_disc_tumor", PESO_DISC_TUMOR)

    hparams_actuales = {
        "LAMBDA_L1": LAMBDA_L1, "LAMBDA_SSIM": LAMBDA_SSIM,
        "LAMBDA_FREQ": LAMBDA_FREQ, "LAMBDA_VGG": LAMBDA_VGG,
        "PESO_TUMOR": PESO_TUMOR, "PESO_DISC_TUMOR": peso_disc_tumor,
        "EMA_DECAY": EMA_DECAY,
    }

    epoca_inicial = 0
    mejor_ssim_tumor = -1.0
    mejor_metrics = None
    warmup_hecho = False
    historial_epocas = []

    # --- Reanudación ---
    if os.path.exists(estado_path):
        print(f"[{nombre_variante} | seed={seed}] Estado previo encontrado -> RETOMANDO entrenamiento")
        estado = torch.load(estado_path, map_location=device, weights_only=False)
        gen.load_state_dict(estado["gen"])
        disc.load_state_dict(estado["disc"])
        opt_gen.load_state_dict(estado["opt_gen"])
        opt_disc.load_state_dict(estado["opt_disc"])
        if ema is not None and estado.get("ema") is not None:
            ema.modelo_ema.load_state_dict(estado["ema"])
        epoca_inicial = estado["epoca"] + 1
        mejor_ssim_tumor = estado["mejor_ssim_tumor"]
        mejor_metrics = estado["mejor_metrics"]
        warmup_hecho = estado["warmup_hecho"]
        historial_epocas = estado.get("historial", [])

        hparams_guardados = estado.get("hparams")
        if hparams_guardados is not None and hparams_guardados != hparams_actuales:
            print("  ⚠️  ADVERTENCIA: hiperparámetros distintos a los guardados en esta corrida:")
            for k in hparams_actuales:
                if hparams_actuales[k] != hparams_guardados.get(k):
                    print(f"      {k}: guardado={hparams_guardados.get(k)} -> actual={hparams_actuales[k]}")

        print(f"  Retomando desde época {epoca_inicial}/{N_EPOCAS} "
              f"(mejor SSIM_tumor hasta ahora: {mejor_ssim_tumor:.4f})")

        if epoca_inicial >= N_EPOCAS:
            print("  Esta corrida ya estaba completa. Devolviendo resultado guardado.")
            return {"variante": nombre_variante, "seed": seed,
                    "ckpt_path": mejor_ckpt_path, **mejor_metrics}

    # --- Warm-up (solo L1) ---
    if config["usar_warmup"] and not warmup_hecho:
        print(f"[{nombre_variante} | seed={seed}] === Warm-up (solo L1, {N_EPOCAS_WARMUP} épocas) ===")
        warmup_generador(gen, train_loader, opt_gen, device, LAMBDA_L1, PESO_TUMOR, N_EPOCAS_WARMUP)
        if ema is not None:
            ema.modelo_ema.load_state_dict(gen.state_dict())
        warmup_hecho = True

    # --- Entrenamiento adversarial ---
    inicio = time.time()
    print(f"[{nombre_variante} | seed={seed}] === Entrenamiento adversarial "
          f"(épocas {epoca_inicial + 1}-{N_EPOCAS}) ===")

    for epoca in range(epoca_inicial, N_EPOCAS):
        if config["usar_disc_tumor_aware"]:
            loss_g, loss_d = entrenar_epoca_disc_tumoraware(
                gen, disc, train_loader, opt_gen, opt_disc, device,
                LAMBDA_L1, PESO_TUMOR, peso_disc_tumor,
                ssim_loss_fn=ssim_loss_fn, freq_loss_fn=freq_loss_fn, vgg_loss_fn=vgg_loss_fn,
                lambda_ssim=LAMBDA_SSIM, lambda_freq=LAMBDA_FREQ, lambda_vgg=LAMBDA_VGG,
            )
        else:
            loss_g, loss_d = entrenar_epoca_ponderada(
                gen, disc, train_loader, opt_gen, opt_disc, device, LAMBDA_L1, PESO_TUMOR
            )

        if ema is not None:
            ema.update(gen)

        fila_historial = {"epoca": epoca + 1, "loss_g": loss_g, "loss_d": loss_d,
                          "ssim_tumor": None, "psnr_tumor": None,
                          "ssim_global": None, "psnr_global": None}

        # Evaluación en validación: época 1 y cada 5 épocas
        if (epoca + 1) % 5 == 0 or epoca == 0:
            modelo_eval = ema.eval_modelo() if ema is not None else gen
            metrics = evaluar_completo(modelo_eval, val_loader, device)
            fila_historial.update(metrics)

            transcurrido = (time.time() - inicio) / 60
            print(f"  Época {epoca + 1}/{N_EPOCAS} ({transcurrido:.1f} min) | "
                  f"SSIM_tumor={metrics['ssim_tumor']:.4f} | PSNR_tumor={metrics['psnr_tumor']:.2f} dB | "
                  f"SSIM_global={metrics['ssim_global']:.4f}")

            if metrics["ssim_tumor"] > mejor_ssim_tumor:
                mejor_ssim_tumor = metrics["ssim_tumor"]
                mejor_metrics = metrics
                torch.save(modelo_eval.state_dict(), mejor_ckpt_path)
                print(f"    ★ Nuevo mejor checkpoint (SSIM_tumor={mejor_ssim_tumor:.4f})")
        else:
            print(f"  Época {epoca + 1}/{N_EPOCAS} | Loss G: {loss_g:.3f} | Loss D: {loss_d:.3f}")

        historial_epocas.append(fila_historial)
        pd.DataFrame(historial_epocas).to_csv(historial_path, index=False)

        # Estado completo para poder reanudar
        torch.save({
            "epoca": epoca,
            "gen": gen.state_dict(),
            "disc": disc.state_dict(),
            "opt_gen": opt_gen.state_dict(),
            "opt_disc": opt_disc.state_dict(),
            "ema": ema.modelo_ema.state_dict() if ema is not None else None,
            "mejor_ssim_tumor": mejor_ssim_tumor,
            "mejor_metrics": mejor_metrics,
            "warmup_hecho": warmup_hecho,
            "hparams": hparams_actuales,
            "historial": historial_epocas,
        }, estado_path)

    return {
        "variante": nombre_variante,
        "seed": seed,
        "ckpt_path": mejor_ckpt_path,
        **mejor_metrics,
    }

## 8. Ejecución del ablation
Entrena cada variante de `VARIANTES_A_ENTRENAR` con cada semilla de `SEEDS`. Por cada corrida: entrena → selecciona el mejor checkpoint en **validación** → evalúa **una sola vez en test**. Los resultados se guardan en CSV tras cada corrida.

- **Reanudable:** las corridas ya completadas (presentes en `resultados_test_por_seed.csv`) se saltan; las interrumpidas retoman desde su último estado guardado.
- Si ya tienes resultados repartidos en varios CSV (p. ej. descargas duplicadas `archivo (3).csv`), ejecuta **antes** el Apéndice A para no reentrenar corridas ya hechas.

In [ ]:
# ──────────────────────────────────────────────────────────────────
# [8.1] Bucle principal (variantes × semillas)
# ──────────────────────────────────────────────────────────────────
# Qué hace : Recorre todas las combinaciones (variante, semilla), salta las
#            ya completadas, entrena, evalúa en test y guarda los CSV de
#            resultados tras cada corrida. Libera memoria de GPU entre
#            corridas.
# Produce  : resultados_val_por_seed.csv, resultados_test_por_seed.csv,
#            checkpoints *_best.pth, historiales *_historial.csv
# ──────────────────────────────────────────────────────────────────
if os.path.exists(PATH_RES_VAL) and os.path.exists(PATH_RES_TEST):
    resultados_val = pd.read_csv(PATH_RES_VAL).to_dict("records")
    resultados_test = pd.read_csv(PATH_RES_TEST).to_dict("records")
    combos_completados = {(r["variante"], r["seed"]) for r in resultados_test}
    print(f"Resultados previos encontrados: {len(combos_completados)} corridas ya completas.")
else:
    resultados_val, resultados_test, combos_completados = [], [], set()

for nombre_variante in VARIANTES_A_ENTRENAR:
    config = CONFIGS[nombre_variante]
    for seed in SEEDS:
        if (nombre_variante, seed) in combos_completados:
            print(f"[{nombre_variante} | seed={seed}] Ya completada -> se salta.")
            continue

        r_val = entrenar_variante(nombre_variante, config, seed, CHECKPOINT_DIR)
        resultados_val.append(r_val)

        r_test = evaluar_en_test(r_val["ckpt_path"])
        r_test.update({"variante": nombre_variante, "seed": seed})
        resultados_test.append(r_test)

        pd.DataFrame(resultados_val).to_csv(PATH_RES_VAL, index=False)
        pd.DataFrame(resultados_test).to_csv(PATH_RES_TEST, index=False)

        gc.collect()
        torch.cuda.empty_cache()

print("\n=== Bucle de ablation completo (o al día, si venías retomando) ===")

## 9. Resultados: tablas y figuras
Todo se lee de los CSV en disco (única fuente de verdad), de modo que esta sección se puede ejecutar en una sesión nueva tras completar las secciones 1–7 (sin volver a entrenar). Las figuras usan etiquetas en inglés, listas para el paper.

In [ ]:
# ──────────────────────────────────────────────────────────────────
# [9.1] Agregación de resultados (media ± desv. estándar)
# ──────────────────────────────────────────────────────────────────
# Qué hace : Lee los resultados de test por semilla, se queda con las 4
#            variantes finales, verifica que estén completas y calcula media
#            y desviación estándar por variante.
# Produce  : df_test_final, resumen_test,
#            resultados_test_final_4variantes.csv, resumen_test_final.csv
# ──────────────────────────────────────────────────────────────────
df_test = pd.read_csv(PATH_RES_TEST)
df_test_final = (df_test[df_test["variante"].isin(FINAL_VARIANTS)]
                 .drop_duplicates(subset=["variante", "seed"])
                 .reset_index(drop=True))

print(df_test_final.groupby("variante")["seed"].apply(lambda s: sorted(s)).reindex(FINAL_VARIANTS))

esperado = len(FINAL_VARIANTS) * len(SEEDS)
assert len(df_test_final) == esperado, f"Se esperaban {esperado} filas, hay {len(df_test_final)}"
assert df_test_final[METRICAS].notna().all().all(), "Hay NaN en las métricas"

df_test_final.to_csv(PATH_TEST_FINAL, index=False)

resumen_test = (df_test_final.groupby("variante")[METRICAS]
                .agg(["mean", "std"]).reindex(FINAL_VARIANTS))
resumen_test.to_csv(PATH_RESUMEN)

print(f"\n=== RESULTADOS FINALES ({len(FINAL_VARIANTS)} variantes, {len(SEEDS)} semillas cada una) ===")
print(resumen_test.round(4))
print("\nGuardado en:", PATH_RESUMEN)

In [ ]:
# ──────────────────────────────────────────────────────────────────
# [9.2] Tabla LaTeX del ablation (booktabs)
# ──────────────────────────────────────────────────────────────────
# Qué hace : Genera la tabla del paper en formato booktabs, con el mejor
#            valor de cada métrica en negrita, y la guarda como .tex.
# Produce  : figuras/tabla_ablation_final.tex
# ──────────────────────────────────────────────────────────────────
def tabla_latex_ablation(resumen_test, orden_variantes=None, n_seeds=3,
                         nombre_archivo="tabla_ablation_final.tex"):
    nombres_metricas = ["SSIM$_{tumor}$", "PSNR$_{tumor}$", "SSIM$_{global}$", "PSNR$_{global}$"]

    if orden_variantes is None:
        orden_variantes = list(resumen_test.index)

    mejores = {m: resumen_test[(m, "mean")].max() for m in METRICAS}

    lineas = []
    lineas.append(r"\begin{table}[h]")
    lineas.append(r"\centering")
    lineas.append(r"\caption{Ablation study results (mean $\pm$ standard deviation over "
                  + str(n_seeds) + r" seeds). Best values in bold.}")
    lineas.append(r"\label{tab:ablation}")
    lineas.append(r"\begin{tabular}{l" + "c" * len(METRICAS) + "}")
    lineas.append(r"\toprule")
    lineas.append(r"\textbf{Model} & " + " & ".join(f"\\textbf{{{n}}}" for n in nombres_metricas) + r" \\")
    lineas.append(r"\midrule")

    for variante in orden_variantes:
        celdas = [variante.replace("_", "\\_")]
        for m in METRICAS:
            media = resumen_test.loc[variante, (m, "mean")]
            std = resumen_test.loc[variante, (m, "std")]
            if "psnr" in m:
                texto = f"{media:.2f} dB $\\pm$ {std:.2f}"
            else:
                texto = f"{media:.4f} $\\pm$ {std:.4f}"
            if abs(media - mejores[m]) < 1e-9:
                texto = f"\\textbf{{{texto}}}"
            celdas.append(texto)
        lineas.append(" & ".join(celdas) + r" \\")

    lineas.append(r"\bottomrule")
    lineas.append(r"\end{tabular}")
    lineas.append(r"\end{table}")

    tabla = "\n".join(lineas)
    out_path = f"{FIG_DIR}/{nombre_archivo}"
    with open(out_path, "w") as f:
        f.write(tabla)
    print(tabla)
    print("\nGuardado en:", out_path)
    return tabla


tabla_latex_ablation(resumen_test, orden_variantes=FINAL_VARIANTS, n_seeds=len(SEEDS))

In [ ]:
# ──────────────────────────────────────────────────────────────────
# [9.3] Figura: curvas de entrenamiento por variante
# ──────────────────────────────────────────────────────────────────
# Qué hace : Para cada variante final (seed=42) dibuja, lado a lado, las
#            pérdidas de G y D por época y el SSIM tumoral en validación.
#            Guarda un PNG por variante.
# Produce  : figuras/curvas_<variante>_seed42_final.png
# ──────────────────────────────────────────────────────────────────
def graficar_curvas_entrenamiento(nombre_variante, seed):
    hist_path = os.path.join(CHECKPOINT_DIR, f"{nombre_variante}_seed{seed}_historial.csv")
    if not os.path.exists(hist_path):
        print(f"No history found for {nombre_variante} seed={seed}")
        return
    df = pd.read_csv(hist_path)

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    # Izquierda: pérdidas del generador y del discriminador
    axes[0].plot(df["epoca"], df["loss_g"], label="Generator Loss")
    axes[0].plot(df["epoca"], df["loss_d"], label="Discriminator Loss")
    axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss")
    axes[0].set_title(f"{nombre_variante} (seed={seed}) — losses")
    axes[0].legend(); axes[0].grid(alpha=0.3)

    # Derecha: SSIM tumoral (solo épocas donde se evaluó)
    df_eval = df.dropna(subset=["ssim_tumor"])
    axes[1].plot(df_eval["epoca"], df_eval["ssim_tumor"], marker="o", label="Tumor SSIM (val)")
    axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Tumor SSIM")
    axes[1].set_title(f"{nombre_variante} (seed={seed}) — tumor SSIM on validation")
    axes[1].legend(); axes[1].grid(alpha=0.3)

    plt.tight_layout()
    out_path = f"{FIG_DIR}/curvas_{nombre_variante}_seed{seed}_final.png"
    plt.savefig(out_path, dpi=150)
    plt.show()
    print("Saved to:", out_path)


for nombre_variante in FINAL_VARIANTS:
    graficar_curvas_entrenamiento(nombre_variante, seed=42)

In [ ]:
# ──────────────────────────────────────────────────────────────────
# [9.4] Figura: barras del ablation
# ──────────────────────────────────────────────────────────────────
# Qué hace : Panel 2×2 (una métrica por gráfico): cada barra es una variante;
#            altura = media sobre las semillas, barra de error = desviación
#            estándar.
# Produce  : figuras/barras_ablation_3seeds_final.png
# ──────────────────────────────────────────────────────────────────
def graficar_barras_ablation(resumen_test, n_seeds=3):
    variantes = list(resumen_test.index)

    fig, axes = plt.subplots(2, 2, figsize=(11, 8))
    axes = axes.flatten()

    for i, metrica in enumerate(METRICAS):
        medias = resumen_test[(metrica, "mean")].values
        stds = resumen_test[(metrica, "std")].values
        axes[i].bar(variantes, medias, yerr=stds, capsize=5)
        axes[i].set_title(metrica)
        axes[i].set_xticks(range(len(variantes)))
        axes[i].set_xticklabels(variantes, rotation=30, ha="right")
        axes[i].grid(alpha=0.3, axis="y")

    plt.suptitle(f"Final ablation — mean ± standard deviation over {n_seeds} seeds")
    plt.tight_layout()
    out_path = f"{FIG_DIR}/barras_ablation_3seeds_final.png"
    plt.savefig(out_path, dpi=150)
    plt.show()
    print("Saved to:", out_path)


graficar_barras_ablation(resumen_test, n_seeds=len(SEEDS))

In [ ]:
# ──────────────────────────────────────────────────────────────────
# [9.5] Figura: estabilidad entre semillas
# ──────────────────────────────────────────────────────────────────
# Qué hace : Un gráfico por variante final con las curvas de SSIM tumoral en
#            validación de TODAS las semillas superpuestas (eje Y
#            compartido), para ver qué tan estable es cada variante.
# Produce  : figuras/curvas_3seeds_superpuestas.png
# ──────────────────────────────────────────────────────────────────
def graficar_curvas_superpuestas(variantes):
    fig, axes = plt.subplots(1, len(variantes), figsize=(4.5 * len(variantes), 4), sharey=True)
    axes = np.atleast_1d(axes)
    for ax, v in zip(axes, variantes):
        for f in sorted(glob.glob(f"{CHECKPOINT_DIR}/{v}_seed*_historial.csv")):
            seed = re.search(r"seed(\d+)", f).group(1)
            d = pd.read_csv(f).dropna(subset=["ssim_tumor"])
            ax.plot(d["epoca"], d["ssim_tumor"], marker="o", ms=3, label=f"seed {seed}")
        ax.set_title(v, fontsize=10); ax.set_xlabel("Epoch"); ax.grid(alpha=0.3); ax.legend()
    axes[0].set_ylabel("Tumor SSIM (val)")
    plt.suptitle(f"Tumor SSIM on validation — {len(SEEDS)} seeds per variant")
    plt.tight_layout()
    out_path = f"{FIG_DIR}/curvas_3seeds_superpuestas.png"
    plt.savefig(out_path, dpi=150)
    plt.show()
    print("Saved to:", out_path)


graficar_curvas_superpuestas(FINAL_VARIANTS)

In [ ]:
# ──────────────────────────────────────────────────────────────────
# [9.6] Figura: comparación cualitativa
# ──────────────────────────────────────────────────────────────────
# Qué hace : Cuadrícula de imágenes: cada fila es un corte de test y las
#            columnas son [T1n entrada] | [salida de cada variante] | [T1ce
#            real]. Filas = n_random cortes aleatorios (semilla fija) + el
#            corte con MAYOR área tumoral. Buscar ese corte recorre todo el
#            test (puede tardar unos minutos).
# Produce  : figuras/grid_comparativo_4variantes.png
# ──────────────────────────────────────────────────────────────────
@torch.no_grad()
def grid_comparativo(dataset, variantes, seed=42, n_random=3, nombre="grid_comparativo_4variantes"):
    # 1) Cargar el mejor checkpoint de cada variante
    gens = {}
    for v in variantes:
        ckpt = f"{CHECKPOINT_DIR}/{v}_seed{seed}_best.pth"
        assert os.path.exists(ckpt), f"No existe: {ckpt}"
        g = GeneradorUNetV2().to(device)
        g.load_state_dict(torch.load(ckpt, map_location=device, weights_only=False))
        g.eval()
        gens[v] = g

    # 2) n_random cortes al azar (semilla fija -> reproducible)
    idx = list(range(len(dataset)))
    random.Random(42).shuffle(idx)
    elegidos = idx[:n_random]

    # 3) Añadir el corte con mayor área tumoral
    print("Searching for the slice with the largest tumor area (may take a few minutes)...")
    areas = [dataset[i][2].sum().item() for i in range(len(dataset))]
    grande = int(np.argmax(areas))
    if grande not in elegidos:
        elegidos.append(grande)

    # 4) Figura: filas = cortes; columnas = entrada + variantes + real
    n_cols = len(variantes) + 2
    fig, axes = plt.subplots(len(elegidos), n_cols, figsize=(2.6 * n_cols, 2.6 * len(elegidos)))
    for fila, i in enumerate(elegidos):
        t1n, t1c, _ = dataset[i]
        x = t1n.unsqueeze(0).to(device)
        imgs = [(t1n[0].numpy() + 1) / 2]                      # [-1, 1] -> [0, 1]
        titulos = ["T1n (input)" + (" — largest area" if i == grande else "")]
        for v in variantes:
            imgs.append((gens[v](x)[0, 0].cpu().numpy() + 1) / 2)
            titulos.append(v)
        imgs.append((t1c[0].numpy() + 1) / 2)
        titulos.append("T1ce (ground truth)")
        for c, (im, t) in enumerate(zip(imgs, titulos)):
            axes[fila, c].imshow(im, cmap="gray", vmin=0, vmax=1)
            axes[fila, c].axis("off")
            if fila == 0 or i == grande:
                axes[fila, c].set_title(t, fontsize=8)

    plt.tight_layout()
    out_path = f"{FIG_DIR}/{nombre}.png"
    plt.savefig(out_path, dpi=150)
    plt.show()
    print("Saved to:", out_path)


grid_comparativo(test_dataset, FINAL_VARIANTS)

In [ ]:
# ──────────────────────────────────────────────────────────────────
# [9.7] Resumen de convergencia por corrida
# ──────────────────────────────────────────────────────────────────
# Qué hace : Para cada variante y semilla: época total, época del mejor SSIM
#            tumoral en validación, y su valor frente al de la última
#            evaluación. Útil para justificar el número de épocas.
# Produce  : resumen_convergencia
# ──────────────────────────────────────────────────────────────────
filas = []
for v in FINAL_VARIANTS:
    for seed in SEEDS:
        d = pd.read_csv(f"{CHECKPOINT_DIR}/{v}_seed{seed}_historial.csv")
        ev = d.dropna(subset=["ssim_tumor"])
        ib = ev["ssim_tumor"].idxmax()
        filas.append({"variante": v, "seed": seed,
                      "epocas_totales": int(d["epoca"].max()),
                      "epoca_mejor": int(ev.loc[ib, "epoca"]),
                      "val_ssim_tumor_mejor": round(ev.loc[ib, "ssim_tumor"], 4),
                      "val_ssim_tumor_ultima": round(ev.iloc[-1]["ssim_tumor"], 4)})

resumen_convergencia = pd.DataFrame(filas)
print(resumen_convergencia.to_string(index=False))
print()
print(resumen_convergencia.groupby("variante")[["epoca_mejor", "val_ssim_tumor_mejor", "val_ssim_tumor_ultima"]]
      .agg(["mean", "std"]).round(4))

## 10. Análisis complementarios
Justificación del peso de parches tumorales en el discriminador (`peso_disc_tumor`): un diagnóstico rápido con peso 1.0 y la comparación 3.0 vs. 1.5 con el protocolo completo.

In [ ]:
# ──────────────────────────────────────────────────────────────────
# [10.1] Diagnóstico rápido: tumor-aware SIN upweighting
# ──────────────────────────────────────────────────────────────────
# Qué hace : Entrena tumor_aware_disc con peso_disc_tumor=1.0 (sin ponderar
#            parches tumorales) durante pocas épocas, para aislar el efecto
#            de ese peso. Es un diagnóstico exploratorio de una sola semilla;
#            no entra en las tablas finales.
# ──────────────────────────────────────────────────────────────────
PESO_DISC_TUMOR_DIAG = 1.0                         # única diferencia respecto a tumor_aware_disc (3.0)
N_EPOCAS_DIAG = 2 if MODO_PRUEBA else 15

seed_everything(42)
train_loader_diag = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                               num_workers=NUM_WORKERS, worker_init_fn=worker_init_fn)

gen_diag = GeneradorUNetV2().to(device)
disc_diag = DiscriminadorTumorAware().to(device)
opt_gen_diag = torch.optim.Adam(gen_diag.parameters(), lr=LR_GEN, betas=(0.5, 0.999))
opt_disc_diag = torch.optim.Adam(disc_diag.parameters(), lr=LR_DISC, betas=(0.5, 0.999))

print("=== Warm-up ===")
warmup_generador(gen_diag, train_loader_diag, opt_gen_diag, device, LAMBDA_L1, PESO_TUMOR, N_EPOCAS_WARMUP)

ema_diag = EMA(gen_diag, decay=EMA_DECAY)
ema_diag.modelo_ema.load_state_dict(gen_diag.state_dict())

for epoca in range(N_EPOCAS_DIAG):
    loss_g, loss_d = entrenar_epoca_disc_tumoraware(
        gen_diag, disc_diag, train_loader_diag, opt_gen_diag, opt_disc_diag, device,
        LAMBDA_L1, PESO_TUMOR, peso_disc_tumor=PESO_DISC_TUMOR_DIAG,
    )
    ema_diag.update(gen_diag)

    if (epoca + 1) % 5 == 0 or epoca == 0:
        metrics = evaluar_completo(ema_diag.eval_modelo(), val_loader, device)
        print(f"Época {epoca + 1}/{N_EPOCAS_DIAG} | Loss G: {loss_g:.3f} | Loss D: {loss_d:.3f} | "
              f"SSIM_tumor={metrics['ssim_tumor']:.4f} | SSIM_global={metrics['ssim_global']:.4f}")

In [ ]:
# ──────────────────────────────────────────────────────────────────
# [10.2] Sensibilidad: peso de parches tumorales 3.0 vs. 1.5
# ──────────────────────────────────────────────────────────────────
# Qué hace : Compara, con el protocolo completo (todas las semillas,
#            evaluación en test), las variantes tumor-aware con peso 3.0
#            frente a 1.5. Requiere haber entrenado las variantes de
#            sensibilidad (sección 2.2). Guarda la comparación como CSV.
# Produce  : comparacion_peso_3.0_vs_1.5.csv
# ──────────────────────────────────────────────────────────────────
pares = [("tumor_aware_disc", "tumor_aware_disc_v2"),
         ("tumor_aware_ssim_freq", "tumor_aware_ssim_freq_v2")]

df_all = pd.read_csv(PATH_RES_TEST)
faltan = sorted({v for par in pares for v in par} - set(df_all["variante"]))

if faltan:
    print("Faltan resultados de:", faltan,
          "\n-> Entrénalas incluyéndolas en VARIANTES_A_ENTRENAR (sección 2.2) y ejecuta 8.1.")
else:
    filas = []
    for v_old, v_new in pares:
        peso_old = CONFIGS[v_old]["peso_disc_tumor"]
        peso_new = CONFIGS[v_new]["peso_disc_tumor"]
        for m in METRICAS:
            a = df_all.loc[df_all["variante"] == v_old, m]
            b = df_all.loc[df_all["variante"] == v_new, m]
            filas.append({
                "variante": v_new.replace("_v2", ""),
                "metrica": m,
                f"peso_{peso_old}": f"{a.mean():.4f} ± {a.std():.4f}",
                f"peso_{peso_new}": f"{b.mean():.4f} ± {b.std():.4f}",
                "delta": round(b.mean() - a.mean(), 4),
            })

    comparacion = pd.DataFrame(filas)
    print(comparacion.to_string(index=False))

    ruta = f"{CHECKPOINT_DIR}/comparacion_peso_3.0_vs_1.5.csv"
    comparacion.to_csv(ruta, index=False)
    print("\nGuardado en:", ruta)

## Apéndice A. Consolidación de resultados previos (opcional)

Solo para quien ya tiene corridas guardadas en **varios CSV** dentro de `CHECKPOINT_DIR` (por ejemplo `resultados_test_por_seed.csv` y `resultados_test_por_seed (3).csv`, típico de descargas/copias duplicadas). Une todos en el archivo único que lee el bucle 8.1, para que no reentrene corridas ya hechas.

Puede ejecutarse en cualquier momento **después de la sección 2** y **antes de 8.1**. Si dos archivos tienen la misma corrida (variante, semilla), se conserva la del archivo modificado más recientemente. Antes de sobrescribir, guarda una copia `.bak`. Si empiezas desde cero, no lo necesitas.

In [ ]:
# ──────────────────────────────────────────────────────────────────
# [A.1] Consolidar CSV de resultados en un único archivo
# ──────────────────────────────────────────────────────────────────
# Qué hace : Une todos los CSV 'resultados_{val,test}_por_seed*.csv' de
#            CHECKPOINT_DIR en PATH_RES_VAL y PATH_RES_TEST, eliminando
#            corridas duplicadas y avisando si sus métricas difieren.
# ──────────────────────────────────────────────────────────────────
def consolidar_csv(patron, destino):
    archivos = sorted(glob.glob(patron), key=os.path.getmtime)      # el más reciente, al final
    if not archivos:
        print(f"Sin archivos para {os.path.basename(patron)}")
        return
    df = pd.concat([pd.read_csv(f) for f in archivos], ignore_index=True)

    dup = df[df.duplicated(["variante", "seed"], keep=False)]
    if len(dup):
        cols = [c for c in METRICAS if c in df.columns]
        dif = dup.groupby(["variante", "seed"])[cols].agg(lambda s: s.max() - s.min()).max(axis=1)
        print(f"  {len(dif)} corridas duplicadas | máx. diferencia de métricas entre copias: {dif.max():.2e}")

    df = df.drop_duplicates(["variante", "seed"], keep="last").reset_index(drop=True)
    if os.path.exists(destino):
        shutil.copy(destino, destino + ".bak")
    df.to_csv(destino, index=False)
    print(f"  {os.path.basename(destino)}: {len(df)} corridas "
          f"({', '.join(sorted(df['variante'].unique()))})")


consolidar_csv(f"{CHECKPOINT_DIR}/resultados_test_por_seed*.csv", PATH_RES_TEST)
consolidar_csv(f"{CHECKPOINT_DIR}/resultados_val_por_seed*.csv",  PATH_RES_VAL)